In [1]:
import sys, os
dir = os.getcwd()

ext = ['', '/..', '/../src', '/../src/nlp', '/../src/models', '/../src/synth']
sys.path += [dir + i for i in ext]

In [2]:
import base64
import requests

IMGBB_API_KEY = '8ae6cb592a734c0c3593f1934b9d90a9'

def upload_im(im, key):
    url = "https://api.imgbb.com/1/upload"
    payload = {
        'key': key,
        'image': base64.b64encode(im)
    }
    response = requests.post(url, data=payload)
    if response.status_code == 200:
        return response.json()['data']['url']
    else:
        raise Exception(f'[{response.status_code}] Failed to upload image.')

In [3]:
from src.nlp.chat import client

class Chat:

    class Entry:
        def __init__(self, role: str, text: str = "", im = None, im_url: str = ''):
            self.role = role
            self.text = text
            if im_url:
                self.im_url = im_url
            elif im:
                self.im_url = upload_im(im, IMGBB_API_KEY)
            else:
                self.im_url = ''

        def to_dict(self) -> dict:
            content = []

            if self.text:
                content.append({
                    'type': 'text',
                    'text': self.text
                })

            if self.im_url:
                content.append({
                    'type': 'image_url',
                    'image_url': {
                        'url': self.im_url
                    }
                })

            obj = {
                'role': self.role,
                'content': content
            }

            return obj

    def __init__(self, client, model='gpt-4o-mini'):
        self.model = model
        self.client = client

    def __call__(self, input: list[Entry] = [], json: bool = False):
        chat = self.client.chat.completions.create(
            model=self.model,
            messages=[entry.to_dict() for entry in input],
            response_format={'type': 'json_object' if json else 'text'},
            #temperature=0
        )
        return chat.choices[0].message.content

In [4]:
def get_im(dir):
    with open(dir, "rb") as im: 
        return im.read()

def get_imi_dir(dir) -> list[str]:
    valid_extensions = {'.png', '.jpg', '.jpeg'}
    files = [f for f in os.listdir(dir) if os.path.splitext(f)[1].lower() in valid_extensions]
    files.sort()
    return [os.path.join(dir, f) for f in files]

In [5]:
keyframes = '/data/transcript_6/frames'
language = """
Coach: So in this situation,you want to realize that you can make a move forward.So after your team passes the ball to you,you want to pass it back immediately.
Coach: Now,you want to get him behind these men in order to penetrate the line of defense over here. And then, recieve the ball.
"""

MODEL = 'o1'
PROMPT = """
You are given a set of set of images from a video where a soccer coach provides instructions on how to behave in a particular situation.
We want to learn this behavior from the coach.
You're tasked to reason about the frames provided to you in the context of the coach's explanation and demonstrations and build a series of subtasks.
The frames are top-down views of a soccer field ordered in chronological order.
Subtasks should be defined for the coach.  

You should return a json with format 

{
    'action': str (An action to take out of ['wait', 'move', 'pass'])
    'task': str (A description of what to do),
    'precondition': str (A description of what condition that needs to be satisfied to execute solving this subtask),
    'termination condition': str (A description of the condition that needs to be satisfied to complete the subtask)
    'frame': dictionary (key: str (either 'precondition' or 'termination condition'), value: int (the index of the image that satisfies either the precondition or the termination condition of this subtask))
}

Note that careful frame selection is important for accurate behavior learning (e.g. when we learn when to make a pass we want to learn from the frame where the pass was made and both players are at the expected locations).
The subtasks should be ordered chronologically too.
"""

imi_dir = get_imi_dir(dir + keyframes)

entries = [
    Chat.Entry(role='system', text=PROMPT),
    Chat.Entry(role='user', text=language)
] + [Chat.Entry(role='user', text=f'Frame index: {idx}', im=get_im(im_dir)) for idx, im_dir in enumerate(imi_dir)]

chat = Chat(client, model=MODEL)
response = chat(entries)
print(response)

{'role': 'system', 'content': [{'type': 'text', 'text': "\nYou are given a set of set of images from a video where a soccer coach provides instructions on how to behave in a particular situation.\nWe want to learn this behavior from the coach.\nYou're tasked to reason about the frames provided to you in the context of the coach's explanation and demonstrations and build a series of subtasks.\nThe frames are top-down views of a soccer field ordered in chronological order.\nSubtasks should be defined for the coach.  \n\nYou should return a json with format \n\n{\n    'action': str (An action to take out of ['wait', 'move', 'pass'])\n    'task': str (A description of what to do),\n    'precondition': str (A description of what condition that needs to be satisfied to execute solving this subtask),\n    'termination condition': str (A description of the condition that needs to be satisfied to complete the subtask)\n    'frame': dictionary (key: str (either 'precondition' or 'termination con

In [6]:
import json
from src.models.scene import Scene
from api.objects.registry import ObjectsAPI

exports = dir + '/data/transcript_7/exports'
parts = [i for i in os.listdir(exports) if i.endswith('.json')]

scene = Scene()

parts.sort()
for j, f in enumerate(parts):
    file = os.path.join(exports, f)
    with open(file) as f:
        data = json.load(f)
        _scene = Scene.from_dict(data, ObjectsAPI)
        scene.extend(_scene)

In [7]:
text = """
success(Optional("```json\n{\n    \"options\": [\n        {\n            \"reasoning\": \"Out of 6 total items, 4 are clean and 2 are in laundry. Percentage of clean items is (4/6)*100 = 66.67%.\",\n            \"title\": \"Clean and Ready\",\n            \"subtitle\": \"67% of your wardrobe is clean and ready to wear.\",\n            \"background\": \"blue\",\n            \"items\": [\"2\", \"3\", \"4\", \"5\"]\n        },\n        {\n            \"reasoning\": \"2 out of 6 items are in laundry. Percentage is (2/6)*100 = 33.33%.\",\n            \"title\": \"Laundry Load\",\n            \"subtitle\": \"33% of your wardrobe needs washing.\",\n            \"background\": \"red\",\n            \"items\": [\"0\", \"1\"]\n        },\n        {\n            \"reasoning\": \"There are 5 different types of clothing items: outerwear, trousers, footwear, sports shirt, and accessories.\",\n            \"title\": \"Wardrobe Variety\",\n            \"subtitle\": \"Your wardrobe includes 5 different types of items.\",\n            \"background\": \"purple\",\n            \"items\": [\"0\", \"1\", \"2\", \"3\", \"4\", \"5\"]\n        },\n        {\n            \"reasoning\": \"There are 2 pairs of trousers: Boggi Milano trousers and ECOALF pants.\",\n            \"title\": \"Double the Trousers\",\n            \"subtitle\": \"You own 2 pairs of trousers.\",\n            \"background\": \"blue\",\n            \"items\": [\"1\", \"3\"]\n        },\n        {\n            \"reasoning\": \"1 out of 6 items is an accessory, making up (1/6)*100 ≈ 16.67% of the wardrobe.\",\n            \"title\": \"Accessorize Smartly\",\n            \"subtitle\": \"17% of your wardrobe consists of accessories.\",\n            \"background\": \"purple\",\n            \"items\": [\"5\"]\n        }\n    ]\n}\n```"), FitOut.OpenAI.Chat.Response(choices: [FitOut.OpenAI.Chat.Response.Choice(index: 0, message: FitOut.OpenAI.Chat.Response.Choice.Message(role: FitOut.OpenAI.Chat.Role.assistant, content: "```json\n{\n    \"options\": [\n        {\n            \"reasoning\": \"Out of 6 total items, 4 are clean and 2 are in laundry. Percentage of clean items is (4/6)*100 = 66.67%.\",\n            \"title\": \"Clean and Ready\",\n            \"subtitle\": \"67% of your wardrobe is clean and ready to wear.\",\n            \"background\": \"blue\",\n            \"items\": [\"2\", \"3\", \"4\", \"5\"]\n        },\n        {\n            \"reasoning\": \"2 out of 6 items are in laundry. Percentage is (2/6)*100 = 33.33%.\",\n            \"title\": \"Laundry Load\",\n            \"subtitle\": \"33% of your wardrobe needs washing.\",\n            \"background\": \"red\",\n            \"items\": [\"0\", \"1\"]\n        },\n        {\n            \"reasoning\": \"There are 5 different types of clothing items: outerwear, trousers, footwear, sports shirt, and accessories.\",\n            \"title\": \"Wardrobe Variety\",\n            \"subtitle\": \"Your wardrobe includes 5 different types of items.\",\n            \"background\": \"purple\",\n            \"items\": [\"0\", \"1\", \"2\", \"3\", \"4\", \"5\"]\n        },\n        {\n            \"reasoning\": \"There are 2 pairs of trousers: Boggi Milano trousers and ECOALF pants.\",\n            \"title\": \"Double the Trousers\",\n            \"subtitle\": \"You own 2 pairs of trousers.\",\n            \"background\": \"blue\",\n            \"items\": [\"1\", \"3\"]\n        },\n        {\n            \"reasoning\": \"1 out of 6 items is an accessory, making up (1/6)*100 ≈ 16.67% of the wardrobe.\",\n            \"title\": \"Accessorize Smartly\",\n            \"subtitle\": \"17% of your wardrobe consists of accessories.\",\n            \"background\": \"purple\",\n            \"items\": [\"5\"]\n        }\n    ]\n}\n```"))], usage: FitOut.OpenAI.Chat.Response.Usage(prompt_tokens: 724, completion_tokens: 2730, total_tokens: 3454)))
success("{\n  \"id\": \"chatcmpl-AvxuQ2eTF4m5RGZS7ufd6stjA2gsh\",\n  \"object\": \"chat.completion\",\n  \"created\": 1738378350,\n  \"model\": \"o1-mini-2024-09-12\",\n  \"choices\": [\n    {\n      \"index\": 0,\n      \"message\": {\n        \"role\": \"assistant\",\n        \"content\": \"```json\\n{\\n    \\\"options\\\": [\\n        {\\n            \\\"reasoning\\\": \\\"Calculated the percentage of dirty items: 2 dirty items (IDs 0 and 1) out of 6 total items. 2/6 = 33%.\\\",\\n            \\\"title\\\": \\\"Clean Sweep Needed!\\\",\\n            \\\"subtitle\\\": \\\"33% of your wardrobe is in need of laundry.\\\",\\n            \\\"background\\\": \\\"red\\\",\\n            \\\"items\\\": [\\\"0\\\", \\\"1\\\"]\\n        },\\n        {\\n            \\\"reasoning\\\": \\\"Determined the proportion of bottomwear: 2 bottomwear items (IDs 1 and 3) out of 6 total items. 2/6 = 33%.\\\",\\n            \\\"title\\\": \\\"Pants Majority\\\",\\n            \\\"subtitle\\\": \\\"33% of your wardrobe consists of bottomwear.\\\",\\n            \\\"background\\\": \\\"blue\\\",\\n            \\\"items\\\": [\\\"1\\\", \\\"3\\\"]\\n        },\\n        {\\n            \\\"reasoning\\\": \\\"Calculated the percentage of clean items: 4 clean items (IDs 2, 3, 4, and 5) out of 6 total items. 4/6 â 67%.\\\",\\n            \\\"title\\\": \\\"Mostly Ready to Wear!\\\",\\n            \\\"subtitle\\\": \\\"67% of your wardrobe is clean and ready to wear.\\\",\\n            \\\"background\\\": \\\"blue\\\",\\n            \\\"items\\\": [\\\"2\\\", \\\"3\\\", \\\"4\\\", \\\"5\\\"]\\n        },\\n        {\\n            \\\"reasoning\\\": \\\"Assessed weather suitability: 5 items are suitable for mild weather out of 6 total items. 5/6 â 83%.\\\",\\n            \\\"title\\\": \\\"Weather Versatility\\\",\\n            \\\"subtitle\\\": \\\"83% of your wardrobe is suitable for mild weather.\\\",\\n            \\\"background\\\": \\\"purple\\\",\\n            \\\"items\\\": [\\\"1\\\", \\\"2\\\", \\\"3\\\", \\\"4\\\", \\\"5\\\"]\\n        },\\n        {\\n            \\\"reasoning\\\": \\\"Identified accessory usage: only 1 accessory item (ID 5) out of 6 total items. 1/6 â 17%.\\\",\\n            \\\"title\\\": \\\"Accessorize Your Style\\\",\\n            \\\"subtitle\\\": \\\"17% of your wardrobe comprises accessories.\\\",\\n            \\\"background\\\": \\\"purple\\\",\\n            \\\"items\\\": [\\\"5\\\"]\\n        }\\n    ]\\n}\\n```\",\n        \"refusal\": null\n      },\n      \"finish_reason\": \"stop\"\n    }\n  ],\n  \"usage\": {\n    \"prompt_tokens\": 724,\n    \"completion_tokens\": 3738,\n    \"total_tokens\": 4462,\n    \"prompt_tokens_details\": {\n      \"cached_tokens\": 0,\n      \"audio_tokens\": 0\n    },\n    \"completion_tokens_details\": {\n      \"reasoning_tokens\": 3264,\n      \"audio_tokens\": 0,\n      \"accepted_prediction_tokens\": 0,\n      \"rejected_prediction_tokens\": 0\n    }\n  },\n  \"service_tier\": \"default\",\n  \"system_fingerprint\": \"fp_f56e40de61\"\n}\n")
"""

print(text)


success(Optional("```json
{
    "options": [
        {
            "reasoning": "Out of 6 total items, 4 are clean and 2 are in laundry. Percentage of clean items is (4/6)*100 = 66.67%.",
            "title": "Clean and Ready",
            "subtitle": "67% of your wardrobe is clean and ready to wear.",
            "background": "blue",
            "items": ["2", "3", "4", "5"]
        },
        {
            "reasoning": "2 out of 6 items are in laundry. Percentage is (2/6)*100 = 33.33%.",
            "title": "Laundry Load",
            "subtitle": "33% of your wardrobe needs washing.",
            "background": "red",
            "items": ["0", "1"]
        },
        {
            "reasoning": "There are 5 different types of clothing items: outerwear, trousers, footwear, sports shirt, and accessories.",
            "title": "Wardrobe Variety",
            "subtitle": "Your wardrobe includes 5 different types of items.",
            "background": "purple",
            "items": ["0"